In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

In [2]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Exploring games data

In [3]:
clean_games_fp = Path(config['games_folder'] + config['games_clean_fp'])

games_df = pd.read_json(clean_games_fp, orient='records')

games_df.head()

,name,rating,updated_at,first_release_date,id,game_modes_Battle Royale,game_modes_Co-operative,game_modes_Massively Multiplayer Online (MMO),game_modes_Multiplayer,game_modes_Single player,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
0,Tom Clancy's The Division 2,74.928458,2026-04-24 00:08:10,1.552608e+09,90099,0,1,1,1,1,...,0,0,0,1,0,0,0,0,0,0
1,The Elder Scrolls Online,71.794992,2026-04-24 00:08:07,1.396570e+09,1081,0,1,1,1,1,...,0,0,0,1,1,0,0,0,0,0
2,BoB: Battle of Bots,NaN,2026-04-23 23:59:03,NaN,354898,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,Far Cry 4,75.787358,2026-04-23 23:57:10,1.416269e+09,6801,0,1,0,1,1,...,0,0,1,1,0,0,0,0,0,0
4,Deathmark,NaN,2026-04-23 23:50:32,NaN,387248,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
#Checking to make sure the id uniquely identifies each row
print("No duplicate IDS") if len(games_df['id'].unique()) == len(games_df) else print("Duplicate IDs found")

No duplicate IDS


In [5]:
#Checking to see if duplicate game names exist
print("No duplicate game names") if len(games_df['name'].unique()) == len(games_df) else print("Duplicate game names found")

Duplicate game names found


In [6]:
#Get all rows with duplicate game names
duplicate_names_df = games_df[games_df.duplicated(subset=['name'], keep=False)].sort_values('name')
duplicate_names_df.head(6)

,name,rating,updated_at,first_release_date,id,game_modes_Battle Royale,game_modes_Co-operative,game_modes_Massively Multiplayer Online (MMO),game_modes_Multiplayer,game_modes_Single player,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
16939,10-Pin Bowling,NaN,2024-11-14 09:27:27,9.334656e+08,92273,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
16938,10-Pin Bowling,NaN,2024-11-14 09:33:09,4.732992e+08,153453,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
6350,15 Minutes,NaN,2026-04-09 19:38:22,1.767571e+09,395433,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
12751,15 Minutes,NaN,2026-02-02 16:52:35,1.761178e+09,355071,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4277,1942,61.392350,2026-04-18 08:37:59,5.031072e+08,272544,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
5536,1942,67.742592,2026-04-13 08:03:19,4.707072e+08,6075,0,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0


In [7]:
#Find columns where duplicates differ (excluding name, rating, updated_at, id)
exclude_cols = {'name', 'rating', 'updated_at', 'id'}
cols_to_check = [col for col in duplicate_names_df.columns if col not in exclude_cols]

i = 0
for game_name in duplicate_names_df['name'].unique():
    game_group = duplicate_names_df[duplicate_names_df['name'] == game_name]
    differing_cols = []
    for col in cols_to_check:
        if len(game_group[col].unique()) > 1:
            differing_cols.append(col)
    if'first_release_date' not in differing_cols:  # Print every 10th game to avoid too much output
        #print(f"{game_name}: {differing_cols}")
        pass
    i+=1

There are many duplicates but they may functionally differ so we'll keep them for now

# Exploring multiplayer modes data

In [27]:
clean_modes_fp = Path(config['multiplayer_modes_folder'] + config['multiplayer_modes_clean_fp'])

modes_df = pd.read_json(clean_modes_fp, orient='records')

modes_df.head()

,id,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen,splitscreenonline
0,9953,92273,False,False,False,NaN,2.0,False,NaN,NaN,22.0,False,NaN
1,1832,7153,True,True,True,2.0,0.0,False,0.0,0.0,12.0,False,NaN
2,7987,57887,False,False,True,0.0,2.0,False,0.0,0.0,49.0,False,NaN
3,7,46076,False,False,False,NaN,30.0,False,NaN,NaN,6.0,False,NaN
4,10207,31256,False,False,False,NaN,NaN,False,NaN,NaN,82.0,False,NaN


In [28]:
dtypes = {
    'id': 'int64',
    'game': 'int64',
    'offlinecoop': 'bool',
    'offlinecoopmax': 'int64',
    'offlinemax': 'int64',
    'onlinecoop': 'bool',
    'onlinecoopmax': 'int64',
    'onlinemax': 'int64',
    'splitscreen': 'bool',
    'splitscreenonline': 'bool'
}

modes_df = modes_df.fillna(-1).astype(dtypes)

If offlinecoop is false, it implies offlinecoopmax should be 0. Same for onlinecoopmax and onlinecoop. Making sure this is the case.

In [44]:
def conflicting_rows(df, bool_col, max_col_1, max_col_2):
    """
    Function to check for conflicting rows based on a boolean column and a maximum value column.

    Inputs: 
    bool_col: column name of the boolean column to check for conflicts
    max_col_1: column name of the first column with maximum values
    max_col_2: column name of the second column with maximum values

    Output:
    Prints out the rows where bool_col is True but max_col is 0 or where bool_col is False but max_col is greater than 0
    """
    conflicts = (((df[bool_col] == True) & (df[max_col_1] <= 0) & (df[max_col_2] <= 0)) |
                  ((df[bool_col] == False) & (df[max_col_1] > 0) & (df[max_col_2] > 0)))
    return conflicts

In [45]:
conflicts_online = conflicting_rows(modes_df, 'onlinecoop', 'onlinecoopmax', 'onlinemax')

online_conflicts_df = modes_df[conflicts_online]
print(str(len(online_conflicts_df)) + " conflicts found for online multiplayer")
online_conflicts_df.head()

2352 conflicts found for online multiplayer


,id,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen,splitscreenonline
59,36,26742,True,False,False,-1,-1,True,-1,-1,82.0,False,True
88,9184,105443,False,False,False,-1,-1,True,-1,-1,6.0,False,True
122,42,51999,False,False,False,-1,-1,True,-1,-1,39.0,False,True
303,9481,114455,False,False,False,-1,-1,True,-1,-1,6.0,False,True
321,11338,136106,True,True,True,-1,-1,False,12,12,48.0,False,True


In [47]:
online_conflicts_with_names = online_conflicts_df.merge(games_df[['id', 'name']], left_on='game', right_on = 'id', how='left')
online_conflicts_with_names.head()

,id_x,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen,splitscreenonline,id_y,name
0,36,26742,True,False,False,-1,-1,True,-1,-1,82.0,False,True,26742.0,TankWars.io
1,9184,105443,False,False,False,-1,-1,True,-1,-1,6.0,False,True,105443.0,Garden Paws
2,42,51999,False,False,False,-1,-1,True,-1,-1,39.0,False,True,51999.0,Dark and Light: Tales of Gaia
3,9481,114455,False,False,False,-1,-1,True,-1,-1,6.0,False,True,114455.0,Pacify
4,11338,136106,True,True,True,-1,-1,False,12,12,48.0,False,True,136106.0,Phantasy Star Online 2: Episode5 Heroes


In [48]:
conflicts_offline = conflicting_rows(modes_df, 'offlinecoop', 'offlinecoopmax', 'offlinemax')

offline_conflicts_df = modes_df[conflicts_offline]
print(str(len(offline_conflicts_df)) + " conflicts found for offline multiplayer")
offline_conflicts_df.head()

1408 conflicts found for offline multiplayer


,id,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen,splitscreenonline
41,8440,83774,False,False,True,0,0,False,0,0,49.0,False,True
205,11601,55057,False,False,False,8,8,True,8,8,6.0,False,True
208,10217,131720,False,False,False,2,2,True,8,100,3.0,True,True
302,8871,107110,False,False,True,-1,-1,False,-1,-1,-1.0,False,True
321,11338,136106,True,True,True,-1,-1,False,12,12,48.0,False,True


In [49]:
offline_conflicts_with_names = offline_conflicts_df.merge(games_df[['id', 'name']], left_on='game', right_on = 'id', how='left')
offline_conflicts_with_names.head()

,id_x,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen,splitscreenonline,id_y,name
0,8440,83774,False,False,True,0,0,False,0,0,49.0,False,True,83774.0,The Jackbox Party Quadpack
1,11601,55057,False,False,False,8,8,True,8,8,6.0,False,True,55057.0,Age of Empires III: Definitive Edition
2,10217,131720,False,False,False,2,2,True,8,100,3.0,True,True,131720.0,Madrun
3,8871,107110,False,False,True,-1,-1,False,-1,-1,-1.0,False,True,107110.0,Balloons Jump
4,11338,136106,True,True,True,-1,-1,False,12,12,48.0,False,True,136106.0,Phantasy Star Online 2: Episode5 Heroes


In [50]:
any_conflicts = conflicts_online | conflicts_offline

any_conflicts_df = modes_df[any_conflicts]
print(str(len(any_conflicts_df)) + " conflicts found for any multiplayer mode")
any_conflicts_df.head()

3389 conflicts found for any multiplayer mode


,id,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen,splitscreenonline
41,8440,83774,False,False,True,0,0,False,0,0,49.0,False,True
59,36,26742,True,False,False,-1,-1,True,-1,-1,82.0,False,True
88,9184,105443,False,False,False,-1,-1,True,-1,-1,6.0,False,True
122,42,51999,False,False,False,-1,-1,True,-1,-1,39.0,False,True
205,11601,55057,False,False,False,8,8,True,8,8,6.0,False,True


In [51]:
offline_conflicts_with_names = offline_conflicts_df.merge(games_df[['id', 'name']], left_on='game', right_on = 'id', how='left')
offline_conflicts_with_names.head()

,id_x,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen,splitscreenonline,id_y,name
0,8440,83774,False,False,True,0,0,False,0,0,49.0,False,True,83774.0,The Jackbox Party Quadpack
1,11601,55057,False,False,False,8,8,True,8,8,6.0,False,True,55057.0,Age of Empires III: Definitive Edition
2,10217,131720,False,False,False,2,2,True,8,100,3.0,True,True,131720.0,Madrun
3,8871,107110,False,False,True,-1,-1,False,-1,-1,-1.0,False,True,107110.0,Balloons Jump
4,11338,136106,True,True,True,-1,-1,False,12,12,48.0,False,True,136106.0,Phantasy Star Online 2: Episode5 Heroes


There are over 4400 modes which contain conflicts for online and offline play. This is a large chunk of data which we'd rather not ignore.

The better option is to make the following assumption: If the data for a given type of play conflicts, assume it doesn't support that type of play. For example, if a game has conflicts for online coop, assume it doesn't support online coop.

In [ ]:
def fix_conflicted_columns(df, bool_col, max_col, max_col_2):
    """
    Function to fix conflicting rows based on a boolean column and a maximum value column.

    Inputs: 
    bool_col: column name of the boolean column to check for conflicts
    max_col: column name of the column with maximum values

    Output:
    Returns a dataframe with the conflicting rows fixed by setting the boolean column to False and the maximum value columns to 0
    """
    new_df = df.copy()
    new_df.loc[conflicting_rows(new_df, bool_col, max_col, max_col_2), bool_col] = False
    new_df.loc[conflicting_rows(new_df, bool_col, max_col, max_col_2), max_col] = 0
    new_df.loc[conflicting_rows(new_df, bool_col, max_col, max_col_2), max_col_2] = 0
    return new_df